In [62]:
#first, exclude non-fires
import arcpy
import os

# --- 1. ENVIRONMENT CONFIGURATION ---
arcpy.env.overwriteOutput = True

# Exact path to your LANDFIRE Raw Events feature class
raw_events = os.path.join(
    base_dir,  
    "LF2024_Public_Raw_Events_CONUS"
)

# Your target project geodatabase and output name
project_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
output_name = "LF_raw_fires_99_24"
output_fc   = os.path.join(project_gdb, output_name)

# --- 2. DEFINE ATTRIBUTE FILTER ---
# Grouping the four wildland fire categories used by LANDFIRE
where_clause = "Event_Type IN ('Wildfire', 'Prescribed Fire', 'Wildland Fire Use', 'Wildland Fire')"

print("Starting LANDFIRE Raw Events processing...")
print(f"Source: {os.path.basename(raw_events)}")
print(f"SQL Filter: {where_clause}")

try:
    print("Extracting fire perimeters and writing to project geodatabase...")
    
    # Run the optimized feature export 
    arcpy.conversion.ExportFeatures(
        in_features=raw_events,
        out_features=output_fc,
        where_clause=where_clause
    )
    
    # Verify the results and report the final count
    result_count = arcpy.management.GetCount(output_fc)[0]
    print("-" * 60)
    print(f"SUCCESS: Created {output_name}")
    print(f"Total fire records extracted: {result_count}")
    print("-" * 60)

except Exception as e:
    print(f"\nCRITICAL ERROR during extraction: {e}\n")

finally:
    # Release file locks and flush memory caches
    arcpy.management.ClearWorkspaceCache(project_gdb)

Starting LANDFIRE Raw Events processing...
Source: LF2024_Public_Raw_Events_CONUS
SQL Filter: Event_Type IN ('Wildfire', 'Prescribed Fire', 'Wildland Fire Use', 'Wildland Fire')
Extracting fire perimeters and writing to project geodatabase...
------------------------------------------------------------
SUCCESS: Created LF_raw_fires_99_24
Total fire records extracted: 600986
------------------------------------------------------------


In [64]:
# ===================================================================
# change projection to match SEFM
# ===================================================================
import arcpy
import os

# --- PATHS ---
project_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
lf_fires    = os.path.join(project_gdb, "LF_raw_fires_99_24")
sefm_events = os.path.join(project_gdb, "SEFM_events_94_24")

output_name = "LF_raw_fires_99_24_projected"
output_fc   = os.path.join(project_gdb, output_name)

print("Checking spatial references...")

sefm_sr = arcpy.Describe(sefm_events).spatialReference
print(f"Target SEFM Projection: {sefm_sr.name}")

lf_sr = arcpy.Describe(lf_fires).spatialReference
print(f"Current Landfire Projection: {lf_sr.name}")

if lf_sr.name != sefm_sr.name:
    print(f"Projections do not match. Reprojecting LANDFIRE data to match SEFM...")
    try:
        # Corrected parameters for arcpy.management.Project
        arcpy.management.Project(
            in_dataset=lf_fires,
            out_dataset=output_fc,
            out_coor_system=sefm_sr
        )
        print(f"SUCCESS: Created {output_name} matching {sefm_sr.name}")
    except Exception as e:
        print(f"Error during projection: {e}")
else:
    print("Projections already match. No transformation needed.")
    if not arcpy.Exists(output_fc):
        arcpy.management.CopyFeatures(lf_fires, output_fc)

arcpy.management.ClearWorkspaceCache(project_gdb)

Checking spatial references...
Target SEFM Projection: AEA_WGS84
Current Landfire Projection: NAD_1983_Contiguous_USA_Albers
Projections do not match. Reprojecting LANDFIRE data to match SEFM...
SUCCESS: Created LF_raw_fires_99_24_projected matching AEA_WGS84


<Result 'true'>

In [65]:
#clip to extent of SE Firemap

import arcpy
import os

# --- PATHS ---
project_gdb  = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
lf_projected = os.path.join(project_gdb, "LF_raw_fires_99_24_projected")
clip_extent  = os.path.join(project_gdb, "extent_Dissolved")

output_name  = "LF_raw_fires_se_clipped"
output_fc    = os.path.join(project_gdb, output_name)

print(f"Clipping projected LANDFIRE features to {os.path.basename(clip_extent)}...")

try:
    # Run the pairwise clip to handle complex overlapping raw polygon geometries
    arcpy.analysis.PairwiseClip(
        in_features=lf_projected,
        clip_features=clip_extent,
        out_feature_class=output_fc
    )
    
    # Check the clipped record count
    result_count = arcpy.management.GetCount(output_fc)[0]
    print("-" * 60)
    print(f"SUCCESS: Created {output_name}")
    print(f"Total fire perimeters remaining in SE extent: {result_count}")
    print("-" * 60)

except Exception as e:
    print(f"Error during clipping process: {e}")

finally:
    # Flush memory caches and clear schema locks
    arcpy.management.ClearWorkspaceCache(project_gdb)

Clipping projected LANDFIRE features to extent_Dissolved...
------------------------------------------------------------
SUCCESS: Created LF_raw_fires_se_clipped
Total fire perimeters remaining in SE extent: 80034
------------------------------------------------------------


In [66]:
#calculate area of each polygon
import arcpy
import os

# --- PATHS ---
project_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
lf_clipped  = os.path.join(project_gdb, "LF_raw_fires_se_clipped")

target_field = "area_ha"

print(f"Checking for '{target_field}' field in {os.path.basename(lf_clipped)}...")

# 1. Check if field already exists; if not, add it
existing_fields = [f.name for f in arcpy.ListFields(lf_clipped)]
if target_field not in existing_fields:
    print(f"Adding field '{target_field}' as a DOUBLE...")
    arcpy.management.AddField(lf_clipped, target_field, "DOUBLE")
else:
    print(f"Field '{target_field}' already exists. Overwriting values...")

print("Calculating polygon areas in Hectares...")

try:
    # 2. Compute geometry attributes
    # "AREA" specifies we want area, and "HECTARES" dictates the unit.
    arcpy.management.CalculateGeometryAttributes(
        in_features=lf_clipped,
        geometry_property=[[target_field, "AREA"]],
        area_unit="HECTARES"
    )
    print(f"SUCCESS: '{target_field}' successfully populated for all features.")

except Exception as e:
    print(f"Error during area calculation: {e}")

finally:
    # Clear caches and release locks
    arcpy.management.ClearWorkspaceCache(project_gdb)

Checking for 'area_ha' field in LF_raw_fires_se_clipped...
Adding field 'area_ha' as a DOUBLE...
Calculating polygon areas in Hectares...
SUCCESS: 'area_ha' successfully populated for all features.


In [67]:
#exclude those less than 0.809 ha

import arcpy
import os

# --- PATHS ---
project_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
lf_clipped  = os.path.join(project_gdb, "LF_raw_fires_se_clipped")

output_name = "LF_raw_fires_se_size_filtered"
output_fc   = os.path.join(project_gdb, output_name)

# --- SQL FILTER ---
# Retain only features where area_ha is greater than or equal to 0.809
where_clause = "area_ha >= 0.809"

print(f"Filtering out LANDFIRE perimeters smaller than 0.809 hectares...")
print(f"SQL Filter: {where_clause}")

try:
    # Run the optimized feature export to create the new clean layer
    arcpy.conversion.ExportFeatures(
        in_features=lf_clipped,
        out_features=output_fc,
        where_clause=where_clause
    )
    
    # Track how many features survived the filter
    original_count = arcpy.management.GetCount(lf_clipped)[0]
    filtered_count = arcpy.management.GetCount(output_fc)[0]
    dropped_count  = int(original_count) - int(filtered_count)
    
    print("-" * 60)
    print(f"SUCCESS: Created {output_name}")
    print(f"Original features:  {original_count}")
    print(f"Filtered features:  {filtered_count} (Dropped {dropped_count} small records)")
    print("-" * 60)

except Exception as e:
    print(f"Error during size filtering: {e}")

finally:
    # Clear workspace locks
    arcpy.management.ClearWorkspaceCache(project_gdb)

Filtering out LANDFIRE perimeters smaller than 0.809 hectares...
SQL Filter: area_ha >= 0.809
------------------------------------------------------------
SUCCESS: Created LF_raw_fires_se_size_filtered
Original features:  80034
Filtered features:  57139 (Dropped 22895 small records)
------------------------------------------------------------


In [68]:
#fix dates

import arcpy
import os
from datetime import datetime

# --- CONFIGURATION ---
arcpy.env.overwriteOutput = True

# Locked-in correct path
fc = os.path.join(base_dir, "output", "LF_raw_fires_se_size_filtered")

# --- STEP 1: CREATE FIELDS ---
print("Checking and adding corrected date fields...")
existing_fields = [f.name for f in arcpy.ListFields(fc)]

if "start_date_corrected" not in existing_fields:
    arcpy.management.AddField(fc, "start_date_corrected", "DATE")

if "end_date_corrected" not in existing_fields:
    arcpy.management.AddField(fc, "end_date_corrected", "DATE")

# --- STEP 2: POPULATE FIELDS ---
print("Populating corrected date fields row-by-row...")

fields = ["Year", "Start_Date", "End_Date", "start_date_corrected", "end_date_corrected"]

with arcpy.da.UpdateCursor(fc, fields) as cur:
    for row in cur:
        # Unpack the row elements into variables
        year, start, end, s_corr, e_corr = row

        # Initialize corrected values with originals
        s_corr = start
        e_corr = end

        # 1. Both missing
        if start is None and end is None and year is not None:
            s_corr = datetime(int(year), 1, 1)
            e_corr = datetime(int(year), 12, 31)

        # 2. Start missing, end present
        elif start is None and end is not None:
            s_corr = end

        # 3. End missing, start present
        elif end is None and start is not None:
            e_corr = start

        # Update the database row using the re-assembled list
        cur.updateRow([year, start, end, s_corr, e_corr])

# Clear database locks
gdb_path = os.path.dirname(fc)
arcpy.management.ClearWorkspaceCache(gdb_path)

print("SUCCESS: Date fields created and successfully updated.")

Checking and adding corrected date fields...
Populating corrected date fields row-by-row...
SUCCESS: Date fields created and successfully updated!


In [69]:
#find for duplicates
import arcpy
import os

# --- PATHS ---
project_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
fc          = os.path.join(project_gdb, "LF_raw_fires_se_size_filtered")

# Temporary output table to store the duplicate tracking data
dup_table   = os.path.join(project_gdb, "LF_duplicate_analysis_table")

print("Analyzing dataset for duplicate geometries and start dates...")

try:
    # Find records matching both Shape (geometry) and start_date_corrected
    # "Shape" checks for identical spatial boundaries
    arcpy.management.FindIdentical(
        in_dataset=fc,
        out_dataset=dup_table,
        fields=["Shape", "start_date_corrected"],
        output_record_option="ONLY_DUPLICATES"  # Only returns records that have a twin
    )
    
    # Check if any duplicates were found
    dup_count = int(arcpy.management.GetCount(dup_table)[0])
    
    if dup_count > 0:
        # Get the unique count of actual fire perimeters affected
        # (FindIdentical lists each match as its own row)
        print("-" * 60)
        print(f"WARNING: Found {dup_count} duplicate records in the data.")
        print(f"Analysis table saved to: {os.path.basename(dup_table)}")
        print("You can open this table in ArcGIS Pro to inspect the matching rows.")
        print("-" * 60)
    else:
        print("-" * 60)
        print("CLEAN: No duplicate geometries with matching start dates found.")
        print("-" * 60)
        # Clean up empty table if none found
        arcpy.management.Delete(dup_table)

except Exception as e:
    print(f"Error checking for duplicates: {e}")

finally:
    arcpy.management.ClearWorkspaceCache(project_gdb)

Analyzing dataset for duplicate geometries and start dates...
------------------------------------------------------------
Analysis table saved to: LF_duplicate_analysis_table
You can open this table in ArcGIS Pro to inspect the matching rows.
------------------------------------------------------------


In [70]:
#scan duplicates, if 1 is wildland fire (unknown), keep the other one
import arcpy
import os
from collections import defaultdict

# --- PATHS ---
project_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
in_fc       = os.path.join(project_gdb, "LF_raw_fires_se_size_filtered")
out_fc      = os.path.join(project_gdb, "LF_raw_fires_se_size_filtered_deduped")

# --- STEP 1: SCAN AND GROUP DUPLICATES ---
print("Scanning dataset to group spatial-temporal duplicates...")
spatial_temporal_groups = defaultdict(list)
fields = ["OID@", "start_date_corrected", "Event_Type", "SHAPE@WKT"]

with arcpy.da.SearchCursor(in_fc, fields) as search_cur:
    for row in search_cur:
        oid, s_date, event_type, wkt_geometry = row
        unique_signature = (wkt_geometry, s_date)
        spatial_temporal_groups[unique_signature].append((oid, event_type))

# --- STEP 2: APPLY SELECTION LOGIC ---
print("Applying priority rules to select master records...")
oids_to_keep = []

for signature, features in spatial_temporal_groups.items():
    if len(features) == 1:
        # No duplicates found, safely keep this standalone feature
        oids_to_keep.append(features[0][0])
    else:
        # Separate specific event types from generic ones
        specific_records = [f for f in features if f[1] != "Wildland Fire"]
        generic_records  = [f for f in features if f[1] == "Wildland Fire"]
        
        if specific_records:
            # Keep the FIRST specific record found
            oids_to_keep.append(specific_records[0][0])
        else:
            # All are "Wildland Fire" -> Keep the FIRST one so we still track 1
            oids_to_keep.append(features[0][0])

# --- STEP 3: EXPORT SELECTED RECORDS TO NEW LAYER ---
print(f"Total unique master features identified: {len(oids_to_keep)}")
print(f"Exporting clean records to new layer: {os.path.basename(out_fc)}...")

# Build a SQL query to select only our master ObjectIDs
# Format: OBJECTID IN (1, 2, 3, ...)
oid_field = arcpy.Describe(in_fc).OIDFieldName
where_clause = f"{oid_field} IN ({','.join(map(str, oids_to_keep))})"

try:
    # Run the optimized feature export using our selection query
    arcpy.conversion.ExportFeatures(
        in_features=in_fc,
        out_features=out_fc,
        where_clause=where_clause
    )
    
    # Verify counts
    initial_count = arcpy.management.GetCount(in_fc)[0]
    final_count   = arcpy.management.GetCount(out_fc)[0]
    dropped_count = int(initial_count) - int(final_count)
    
    print("-" * 60)
    print("SUCCESS: Deduplicated layer created successfully.")
    print(f"Original Layer Count:    {initial_count}")
    print(f"Deduplicated Layer Count: {final_count}")
    print(f"Redundant Rows Excluded:  {dropped_count}")
    print("-" * 60)

except Exception as e:
    print(f"Error during feature export: {e}")

finally:
    # Clear database memory locks
    arcpy.management.ClearWorkspaceCache(project_gdb)

Scanning dataset to group spatial-temporal duplicates...
Applying priority rules to select master records...
Total unique master features identified: 52757
Exporting clean records to new layer: LF_raw_fires_se_size_filtered_deduped...
------------------------------------------------------------
SUCCESS: Deduplicated layer created successfully!
Original Layer Count:    57139
Deduplicated Layer Count: 52757
Redundant Rows Excluded:  4382
------------------------------------------------------------
